In [1]:
#Importing libraries for the dataset
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

In [2]:
dataset=pd.read_csv("Preprocessed_Border.csv")

In [3]:
dataset.isnull().sum()

sensor_id                       0
timestamp_hour                  0
motion_intensity                0
vibration_level                 0
thermal_signature               0
acoustic_level                  0
object_speed_kmph               0
distance_m                      0
altitude_m                      0
water_pressure                  0
signal_noise_ratio              0
weight                          0
route_mode_Ground               0
route_mode_Vehicle Route        0
route_mode_Water Route          0
weather_condition_Fog           0
weather_condition_Rain          0
weather_condition_Storm         0
terrain_type_Mountain           0
terrain_type_Open Field         0
terrain_type_Riverbank          0
sensor_status_Degraded          0
target_class_Animal             0
target_class_False Alarm        0
target_class_Human              0
target_class_Vehicle            0
target_class_Water Intrusion    0
dtype: int64

In [4]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [5]:
indep_X=dataset.drop(['target_class_Animal','target_class_False Alarm','target_class_Human','target_class_Vehicle','target_class_Water Intrusion'],axis=1)

In [6]:
dep_Y=dataset[['target_class_Animal','target_class_False Alarm','target_class_Human','target_class_Vehicle','target_class_Water Intrusion']].idxmax(axis=1)

In [7]:
X_train,X_test,y_train,y_test=train_test_split(indep_X,dep_Y,test_size=0.3,random_state=0)

In [8]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
log_rfe = RFE(estimator=RF,n_features_to_select=3)
log_fit = log_rfe.fit(indep_X, dep_Y)
log_rfe_feature=log_fit.transform(indep_X)
mask=log_fit.get_support()
selected_features=indep_X.columns[mask].tolist()
print(selected_features)

['sensor_id', 'object_speed_kmph', 'weight']


In [10]:
classifier = SVC(kernel = 'linear', random_state = 0)
classifier.fit(X_train, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [11]:
y_pred=classifier.predict(X_test)# Eavluate test set input and predict the values
y_pred

array(['target_class_Human', 'target_class_Human',
       'target_class_Water Intrusion', 'target_class_Human',
       'target_class_Water Intrusion', 'target_class_Vehicle',
       'target_class_Vehicle', 'target_class_Animal',
       'target_class_Animal', 'target_class_Animal',
       'target_class_False Alarm', 'target_class_Animal',
       'target_class_Animal', 'target_class_Vehicle',
       'target_class_Animal', 'target_class_Vehicle',
       'target_class_Vehicle', 'target_class_False Alarm',
       'target_class_Animal', 'target_class_Animal',
       'target_class_Animal', 'target_class_Water Intrusion',
       'target_class_Human', 'target_class_Animal',
       'target_class_Vehicle', 'target_class_Human', 'target_class_Human',
       'target_class_Animal', 'target_class_Animal',
       'target_class_Vehicle', 'target_class_Human', 'target_class_Human',
       'target_class_Human', 'target_class_Animal', 'target_class_Human',
       'target_class_Animal', 'target_class_Anima

In [12]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)
print(cm)

[[37  0  0  0  0]
 [ 0  8  0  0  0]
 [ 1  0 22  0  0]
 [ 0  0  0 20  0]
 [ 0  0  0  0 17]]


In [13]:
from sklearn.metrics import classification_report
clf_report=classification_report (y_test,y_pred)
print(clf_report)

                              precision    recall  f1-score   support

         target_class_Animal       0.97      1.00      0.99        37
    target_class_False Alarm       1.00      1.00      1.00         8
          target_class_Human       1.00      0.96      0.98        23
        target_class_Vehicle       1.00      1.00      1.00        20
target_class_Water Intrusion       1.00      1.00      1.00        17

                    accuracy                           0.99       105
                   macro avg       0.99      0.99      0.99       105
                weighted avg       0.99      0.99      0.99       105



In [22]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,classifier.predict_proba(X_test)[:,1])

AttributeError: This 'SVC' has no attribute 'predict_proba'